# 02 — Variable Validation (UKDA TAB waves 2018–2025)

This notebook validates the structure of the UK Cyber Security Breaches Survey (CSBS) wave files (2018–2025) using the original UKDA tab-delimited releases. The objective is to establish which variables are comparable across all waves (“stable core”) and which variables exist only in later years (“availability windows”). This prevents longitudinal artefacts caused by questionnaire revisions, routing logic, and category expansions.

This notebook is methodological rather than analytical. It does not attempt to explain trends or infer causality. Its outputs define the permissible variable sets for the four poster narratives and provide an auditable record of inclusion/exclusion decisions.

----
#### 1. Imports and project paths

This cell loads required libraries and defines project-relative paths. The notebook is stored inside `CSLS_Project/notebooks/`, so the project root is set as the parent directory of the current working directory. Raw `.tab` files in `data_raw/` are treated as immutable canonical sources. All validation artefacts created in this notebook are written to `outputs/tables/` to support reproducibility and later reporting.

In [1]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data_raw"
OUT_TABLES = PROJECT_ROOT / "outputs" / "tables"
OUT_TABLES.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DIR:", RAW_DIR)
print("OUT_TABLES:", OUT_TABLES)

PROJECT_ROOT: C:\Users\toby\OneDrive - Node9\Documents\College & Uni\Data Science Degree\Year Three\WBP\CSLS_Project
RAW_DIR: C:\Users\toby\OneDrive - Node9\Documents\College & Uni\Data Science Degree\Year Three\WBP\CSLS_Project\data_raw
OUT_TABLES: C:\Users\toby\OneDrive - Node9\Documents\College & Uni\Data Science Degree\Year Three\WBP\CSLS_Project\outputs\tables


-----
#### 2. Wave registry and file existence checks

This cell creates a year-to-file mapping for 2018–2025 and confirms that all expected raw wave files exist before any processing occurs. This avoids silent partial runs (e.g., analysing fewer waves than intended), which would invalidate longitudinal comparisons.

In [2]:
years = list(range(2018, 2026))
tab_files = {year: RAW_DIR / f"{year}.tab" for year in years}

missing = [year for year, p in tab_files.items() if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing tab files for years: {missing}. Check RAW_DIR={RAW_DIR}")

print("Example file:", tab_files[2018])

Example file: C:\Users\toby\OneDrive - Node9\Documents\College & Uni\Data Science Degree\Year Three\WBP\CSLS_Project\data_raw\2018.tab


-----
#### 3. Column normalisation policy

UKDA tab files may contain subtle header differences (whitespace, casing, or byte-order marks). To make cross-wave comparison reliable, this notebook normalises column names by stripping whitespace, removing BOM artefacts, and converting to lowercase. This does not alter the substantive content of the data; it ensures that column comparisons are not distorted by formatting inconsistencies.

In [3]:
def normalise_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.replace("\ufeff", "", regex=False)
        .str.strip()
        .str.lower()
    )
    return df

----
#### 4. Load all waves and add wave identifier (`year`)

This cell loads each UKDA `.tab` wave file, applies the column normalisation policy, and adds a `year` column. The `year` column is not part of the UKDA raw data; it is a derived identifier added during import to enable stacking and year-based grouping in later analysis. Because it is synthetic, it is excluded from stable-core computations.

In [4]:
data = {}
columns_by_year = {}

for year, path in tab_files.items():
    df = pd.read_csv(path, sep="\t", low_memory=False)
    df = normalise_columns(df)
    df["year"] = year  # derived wave identifier
    data[year] = df
    columns_by_year[year] = set(df.columns)

print({y: len(cols) for y, cols in columns_by_year.items()})
print("Year column present in all waves:", all("year" in cols for cols in columns_by_year.values()))

{2018: 462, 2019: 462, 2020: 313, 2021: 421, 2022: 447, 2023: 568, 2024: 499, 2025: 528}
Year column present in all waves: True


----
#### 5. Compute the stable longitudinal core (excluding derived fields)

This cell computes the intersection of variable names across all waves, excluding the derived `year` identifier. The result is the empirical stable core: variables that exist in every wave file and can therefore support full-period 2018–2025 comparisons without harmonisation. Variables outside this core may still be usable in later-wave posters, but only within explicitly declared availability windows.

In [5]:
core_sets = [cols - {"year"} for cols in columns_by_year.values()]
stable_all_years = set.intersection(*core_sets)

print("Stable core variable count (2018–2025):", len(stable_all_years))

Stable core variable count (2018–2025): 172


----
#### 6. Validate segmentation hierarchy variables

This cell validates the presence and basic stability of segmentation variables used throughout the project. The top-level segmentation is organisation type (e.g., business, charity, education), and business-only sub-segmentation uses business size and sector. These checks ensure that later comparisons do not mix incompatible denominators or routed fields.

In [6]:
key_seg_vars = ["samptype", "sizeb", "sector_comb2", "weight"]

for var in key_seg_vars:
    print(var, {y: (var in columns_by_year[y]) for y in years})

samptype {2018: True, 2019: True, 2020: True, 2021: True, 2022: True, 2023: True, 2024: True, 2025: True}
sizeb {2018: True, 2019: True, 2020: True, 2021: True, 2022: True, 2023: True, 2024: True, 2025: True}
sector_comb2 {2018: True, 2019: True, 2020: True, 2021: True, 2022: True, 2023: True, 2024: True, 2025: True}
weight {2018: True, 2019: True, 2020: True, 2021: True, 2022: True, 2023: True, 2024: True, 2025: True}


----
#### 7. Organisation type (`samptype`) distribution sanity check

This cell inspects the distribution of the organisation type variable by wave. The intent is to confirm that coding is present and broadly consistent across years. This is not used to draw conclusions; it is a structural validity check that the segmentation key behaves as expected.

In [7]:
for y in years:
    df = data[y]
    if "samptype" in df.columns:
        print("\n", y)
        print(df["samptype"].value_counts(dropna=False).head(10))


 2018
samptype
1    1519
2     569
Name: count, dtype: int64

 2019
samptype
1    1615
2     465
Name: count, dtype: int64

 2020
samptype
1    1374
2     311
3     215
Name: count, dtype: int64

 2021
samptype
1    1448
2     458
3     378
Name: count, dtype: int64

 2022
samptype
1    1265
3     490
2     402
Name: count, dtype: int64

 2023
samptype
1    2413
2    1024
3     554
Name: count, dtype: int64

 2024
samptype
1    2000
2    1004
3     430
Name: count, dtype: int64

 2025
samptype
1    2180
2    1081
3     574
Name: count, dtype: int64


----
#### 8. Business size routing check (`sizeb`)

Business size is expected to apply within businesses and be routed or missing for non-business respondents. This cell checks whether `sizeb` behaves like a routed variable by comparing its distribution across organisation types. If size is populated outside the business subset, denominators and segmentation logic must be re-examined before analysis.

In [8]:
for y in years:
    df = data[y]
    if {"samptype", "sizeb"}.issubset(df.columns):
        print("\n", y)
        print(df.groupby("samptype")["sizeb"].value_counts(dropna=False).head(10))


 2018
samptype  sizeb
1          1       655
           2       349
           3       263
           4       252
2          2       251
           1       131
           3       121
           4        63
          -97        3
Name: count, dtype: int64

 2019
samptype  sizeb
1          1       770
           2       330
           3       301
           4       214
2          2       200
           3       116
           1       101
           4        44
          -97        4
Name: count, dtype: int64

 2020
samptype  sizeb
1          1       644
           2       286
           3       223
           4       221
2          2       132
           1        87
           3        52
           4        38
          -97        2
3          3        94
Name: count, dtype: int64

 2021
samptype  sizeb
1          1       746
           2       275
           3       219
           4       208
2          2       182
           1       128
           3        96
           4        49
  

----
#### 9. Sector variable (`sector_comb2`) presence and stability

Sector analysis is only valid within the business subset. This cell checks whether `sector_comb2` exists across waves and inspects category distributions to identify possible coding drift. Any drift must be harmonised explicitly or sector comparisons must be restricted to stable windows.

In [9]:
for y in years:
    df = data[y]
    if "sector_comb2" in df.columns:
        print("\n", y)
        print(df["sector_comb2"].value_counts(dropna=False).head(20))


 2018
sector_comb2
      569
10    217
1     150
12    148
2     145
9     141
6     119
4     106
5     105
7     101
8      99
11     94
3      94
Name: count, dtype: int64

 2019
sector_comb2
      514
10    221
1     179
9     172
12    140
2     132
11    120
6     119
5     117
8     102
3      95
4      90
7      79
Name: count, dtype: int64

 2020
sector_comb2
      552
10    161
1     161
9     143
12    143
2     142
5     121
8     120
7     117
6     109
4      63
11     43
3      25
Name: count, dtype: int64

 2021
sector_comb2
      865
10    229
1     203
9     159
12    157
2     120
7     109
6     108
8      95
5      94
4      78
11     48
3      19
Name: count, dtype: int64

 2022
sector_comb2
      914
10    155
9     146
1     144
8     136
7     120
2     113
6     102
12     85
5      82
4      64
13     40
11     40
3      16
Name: count, dtype: int64

 2023
sector_comb2
      1728
10     345
9      285
1      240
2      239
6      236
12     206
5      178
8 

----
#### 10. Governance variable family stability

Governance maturity is operationalised using stable governance indicators. This cell identifies governance-related variable families (`priority`, `manage*`, `policy*`) and computes their intersections across waves. Only the stable subsets are eligible for the full 2018–2025 governance index used in Poster 1.

In [10]:
manage_by_year = {y: {c for c in (columns_by_year[y] - {"year"}) if c.startswith("manage")} for y in years}
policy_by_year = {y: {c for c in (columns_by_year[y] - {"year"}) if c.startswith("policy")} for y in years}

stable_manage = set.intersection(*manage_by_year.values()) if years else set()
stable_policy = set.intersection(*policy_by_year.values()) if years else set()

print("priority present all years:", all("priority" in (columns_by_year[y] - {"year"}) for y in years))
print("Stable manage* count:", len(stable_manage))
print("Stable policy* count:", len(stable_policy))
print("Example manage*:", sorted(stable_manage)[:20])
print("Example policy*:", sorted(stable_policy)[:20])

priority present all years: True
Stable manage* count: 6
Stable policy* count: 7
Example manage*: ['manage1', 'manage2', 'manage3', 'manage4', 'manage6', 'manage7']
Example policy*: ['policy1', 'policy2', 'policy3', 'policy4', 'policy5', 'policy8', 'policy9']


----
#### 11. Control adoption variables (10 Steps / Essentials)

This cell identifies variables related to control adoption frameworks (e.g., 10 Steps, Cyber Essentials). It first searches the stable core for control-related fields and then reports wave-by-wave counts to detect whether some controls only appear in later waves. These results determine whether a control adoption index can be defined for 2018–2025, or whether it must be limited to a later window.

In [11]:
def find_control_cols(cols):
    cols = [c for c in cols if c != "year"]
    hits = [c for c in cols if c.lower().startswith("step") or "10steps" in c.lower() or "essential" in c.lower()]
    return sorted(hits)

stable_controls = find_control_cols(stable_all_years)
print("Control-related cols in stable core:", stable_controls)

for y in years:
    hits = find_control_cols(columns_by_year[y])
    print(y, len(hits))

Control-related cols in stable core: ['allessentials', 'any10steps', 'step1', 'step10', 'step2', 'step3', 'step4', 'step5', 'step6', 'step7', 'step8', 'step9', 'sum10steps']
2018 13
2019 13
2020 13
2021 13
2022 13
2023 13
2024 13
2025 13


----
#### 12. Breach type variables (`type*`) and category expansion

Breach experience is derived from multi-response breach-type indicators (`type*`). This cell identifies the stable intersection of breach-type variables across all waves and quantifies any additional breach categories introduced in specific years. This prevents a longitudinal artefact where later waves appear to have “more types” simply because new categories were added to the questionnaire. Poster 2 will use stable breach types for full-period trends and may show expanded categories only within clearly labelled later-wave windows.

In [12]:
type_by_year = {y: {c for c in (columns_by_year[y] - {"year"}) if c.startswith("type")} for y in years}
stable_type = set.intersection(*type_by_year.values())

print("Stable type* count:", len(stable_type))
print("Stable type* vars:", sorted(stable_type))

for y in years:
    extra = type_by_year[y] - stable_type
    print(y, "extra type*:", len(extra))

Stable type* count: 13
Stable type* vars: ['type1', 'type10', 'type11', 'type12', 'type2', 'type3', 'type4', 'type5', 'type6', 'type7', 'type8', 'type9', 'typex']
2018 extra type*: 0
2019 extra type*: 0
2020 extra type*: 1
2021 extra type*: 3
2022 extra type*: 4
2023 extra type*: 4
2024 extra type*: 7
2025 extra type*: 7


----
#### 13. Reporting behaviour variables (`report*`)

This cell identifies reporting-related variables (`report*`) and computes which are stable across 2018–2025. Reporting behaviour supports response-maturity narratives and can be more interpretable than incident prevalence alone. Stable reporting variables are eligible for the full-period posters; later-wave reporting variables may be used only within clearly declared time windows.

In [13]:
report_by_year = {y: {c for c in (columns_by_year[y] - {"year"}) if c.startswith("report")} for y in years}
stable_report = set.intersection(*report_by_year.values())

print("Stable report* count:", len(stable_report))
print("Stable report* vars:", sorted(stable_report))

Stable report* count: 21
Stable report* vars: ['reporta', 'reportb1', 'reportb10', 'reportb11', 'reportb12', 'reportb13', 'reportb14', 'reportb15', 'reportb16', 'reportb17', 'reportb18', 'reportb2', 'reportb21', 'reportb22', 'reportb24', 'reportb3', 'reportb5', 'reportb6', 'reportb7', 'reportb8', 'reportb9']


----
#### 14. Training availability window (`trained`)

Training adoption is not assumed to be measured in all waves. This cell checks which years contain the training variable and defines its availability window. Any training-related poster will be limited to this window and will interpret training as an association rather than a causal driver of breach reduction, due to reactive adoption and reporting maturity effects.

In [14]:
training_years = [y for y in years if "trained" in (columns_by_year[y] - {"year"})]
print("Training years:", training_years)

Training years: [2021, 2022, 2023, 2024, 2025]


----
#### Save validation artefacts for reproducibility

This cell creates and saves a compact presence summary for key metric families and segmentation variables. The resulting table provides an auditable record of variable availability by wave and supports later methodological write-up (A3) and poster footnotes. Saving these artefacts reduces the risk of “silent” changes in variable sets as the project develops.

In [15]:
def count_prefix(cols, prefix):
    return len([c for c in cols if c.startswith(prefix)])

rows = []
for y in years:
    cols = columns_by_year[y] - {"year"}
    rows.append({
        "year": y,
        "n_vars_total": len(cols),
        "has_samptype": "samptype" in cols,
        "has_sizeb": "sizeb" in cols,
        "has_sector_comb2": "sector_comb2" in cols,
        "has_priority": "priority" in cols,
        "has_trained": "trained" in cols,
        "n_manage_vars": count_prefix(cols, "manage"),
        "n_policy_vars": count_prefix(cols, "policy"),
        "n_type_vars": count_prefix(cols, "type"),
        "n_report_vars": count_prefix(cols, "report"),
    })

presence_df = pd.DataFrame(rows).sort_values("year")
presence_path = OUT_TABLES / "variable_presence_summary.csv"
presence_df.to_csv(presence_path, index=False)

print("Saved:", presence_path)
presence_df

Saved: C:\Users\toby\OneDrive - Node9\Documents\College & Uni\Data Science Degree\Year Three\WBP\CSLS_Project\outputs\tables\variable_presence_summary.csv


,year,n_vars_total,has_samptype,has_sizeb,has_sector_comb2,has_priority,has_trained,n_manage_vars,n_policy_vars,n_type_vars,n_report_vars
0,2018,461,True,True,True,True,False,7,9,13,30
1,2019,461,True,True,True,True,False,7,9,13,31
2,2020,312,True,True,True,True,False,7,9,14,35
3,2021,420,True,True,True,True,True,7,12,16,43
4,2022,446,True,True,True,True,True,8,12,17,48
5,2023,567,True,True,True,True,True,8,12,17,46
6,2024,498,True,True,True,True,True,8,11,20,25
7,2025,527,True,True,True,True,True,8,11,20,25
